In [5]:
from LIBS_PESSOAL.GustavoAplicaMetodoDeWelchPLOT import MetodoDeWelch
import pandas as pd
import pvlib
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

In [6]:
path_date = './DADOS/Dados eficiência da Usina Petrolina.parquet'
df_full = pd.read_parquet(path_date, columns=['Time', 'Potência AC total (kW)', 'POA (W/m²)'])
df_full.rename(columns={'Potência AC total (kW)':'P', 'POA (W/m²)':'G'}, inplace=True)
st = df_full.set_index('Time')
st.index = st.index.tz_localize('America/Recife')
idx_completo = pd.date_range(st.index.min().floor('D'), st.index.max().ceil('D'), freq='1min', inclusive='left')
st = st.reindex(idx_completo)
st.index.name = 'Time'
st.sort_index(ascending=True, inplace=True)

In [ ]:
Local=pvlib.location.Location(latitude=-9.398611, longitude=-40.500000, altitude=0, tz='America/Recife', name='Usina Solar Petrolina')
solpos=Local.get_solarposition(st.index)
ceu_claro=Local.get_clearsky(st.index)
dni_extra = pvlib.irradiance.get_extra_radiation(st.index)

# 3. MÓDULO DE TRACKER (Rastreador de Eixo Único)
# Assumindo o padrão de mercado: eixo Norte-Sul, inclinação 0 (paralelo ao solo), girando ±60 graus
tracker_data = pvlib.tracking.singleaxis(
    apparent_zenith=solpos['apparent_zenith'],
    solar_azimuth=solpos['azimuth'],
    axis_tilt=0, 
    axis_azimuth=0, 
    max_angle=60)

# 4. Transposição de Perez para o painel em movimento
poa_tracker = pvlib.irradiance.get_total_irradiance(
    surface_tilt=tracker_data['surface_tilt'],     # A inclinação muda a cada minuto!
    surface_azimuth=tracker_data['surface_azimuth'], # O azimute muda a cada minuto!
    solar_zenith=solpos['apparent_zenith'],
    solar_azimuth=solpos['azimuth'],
    dni=ceu_claro['dni'],
    ghi=ceu_claro['ghi'],
    dhi=ceu_claro['dhi'],
    dni_extra=dni_extra,
    albedo=0.2,
    model='perez')

st['Zenith'] = solpos['apparent_zenith']
st['ClearSky_Tracker'] = poa_tracker['poa_global']
st['G'] = np.where(st['Zenith'] > 88, 0, st['G'])

st['G_NaNs__Trocado_Por_Linear'] = st['G'].interpolate(method='linear')
st['G_NaNs__Trocado_Por_Zeros'] = st['G'].fillna(0)

st['Indice_Kt'] = np.where(st['Zenith'] <= 88, st['G'] / st['ClearSky_Tracker'], np.nan)
st['Indice_Kt'] = st['Indice_Kt'].clip(lower=0, upper=1.2)

bins = [-np.inf, 0.3, 0.6, 0.9, np.inf]
labels = [1, 2, 3, 4]

# Criando a coluna de Estados discretos apenas onde temos dados válidos
st['Estado_Kt'] = pd.cut(st['Indice_Kt'], bins=bins, labels=labels)

# Isolamos apenas os dados válidos para treinar a Cadeia de Markov
st_validos = st.dropna(subset=['Estado_Kt']).copy()

# Criamos uma coluna com o estado deslocado em 1 minuto (t-1)
st_validos['Estado_Kt_Anterior'] = st_validos['Estado_Kt'].shift(1)
# Removemos o primeiro NaN gerado pelo shift
st_validos.dropna(subset=['Estado_Kt_Anterior'], inplace=True)

# Construção da Matriz de Transição P (frequência cruzada normalizada pelas linhas)
matriz_P = pd.crosstab(
    st_validos['Estado_Kt_Anterior'], 
    st_validos['Estado_Kt'], 
    normalize='index') # Garante que a soma das probabilidades de cada linha seja 1.0

print("Matriz de Transição P concluída:")

# Dicionário guardando os valores históricos reais de Kt para cada estado
valores_intra_estado = {
    estado: st_validos[st_validos['Estado_Kt'] == estado]['Indice_Kt'].values 
    for estado in labels}

# 1. Preparação das colunas de saída (para não sobrescrever os dados originais)
st['Kt_Imputado'] = st['Indice_Kt'].copy()
st['Estado_Imputado'] = st['Estado_Kt'].copy()

# 2. Identificação estrita dos Gaps
mascara_falhas = st['Kt_Imputado'].isna() & (st['Zenith'] < 88)
indices_falhas = st[mascara_falhas].index

print(f"Iniciando imputação de {len(indices_falhas)} minutos falhos...")

# 3. O Loop de Markov (Iteração Cronológica)
for t in indices_falhas:
    t_minus_1 = t - pd.Timedelta(minutes=1)
    
    # Verifica se o instante anterior possui um estado válido (mesmo que tenha sido imputado no loop anterior)
    if t_minus_1 in st.index and pd.notna(st.loc[t_minus_1, 'Estado_Imputado']):
        estado_atual = st.loc[t_minus_1, 'Estado_Imputado']
        
        # A. Transição de Estado (A roleta de Markov)
        # Extrai as probabilidades da linha correspondente na Matriz P
        probabilidades_transicao = matriz_P.loc[estado_atual].values
        
        # Sorteia o próximo estado meteorológico
        novo_estado = np.random.choice(labels, p=probabilidades_transicao)
        
    else:
        # Caso de Borda (ex: primeiro minuto do dia é uma falha)
        # Como não temos o t-1, usamos a Frequência Incondicional (probabilidade global de cada estado)
        prob_global = st_validos['Estado_Kt'].value_counts(normalize=True).sort_index().values
        novo_estado = np.random.choice(labels, p=prob_global)
    
    # B. Amostragem Intra-Estado (O valor físico de Kt)
    # Sorteia aleatoriamente um valor do pool histórico que pertence exclusivamente ao novo_estado
    novo_kt = np.random.choice(valores_intra_estado[novo_estado])
    
    # C. Injeção no DataFrame
    st.loc[t, 'Estado_Imputado'] = novo_estado
    st.loc[t, 'Kt_Imputado'] = novo_kt

print("Imputação estocástica concluída.")

# 4. Reconstrução do Sinal Físico (Irradiância Global)
# G_t = Kt * G_clear, garantindo que à noite a irradiância seja zero
st['G_Imputado'] = np.where(st['Zenith'] < 88, st['Kt_Imputado'] * st['ClearSky_Tracker'], 0)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

config_visual = {
    'figure.figsize': (7, 4),
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'mathtext.fontset': 'stix'}
        
with plt.rc_context(config_visual):
    nomes_estados = ['Fechado', 'Nublado', 'Encoberto', 'Limpo']
    
    # AJUSTE 1: Atribuímos o gráfico à variável 'ax' e removemos o cbar_kws daqui
    ax = sns.heatmap(matriz_P, 
                     annot=True, 
                     fmt=".3f",      
                     cmap='YlOrRd', 
                     xticklabels=nomes_estados,
                     yticklabels=nomes_estados)
    
    # AJUSTE 2: Acessamos a barra de cores gerada pelo 'ax'
    cbar = ax.collections[0].colorbar
    # Definimos o título da barra e usamos 'labelpad' para afastar o texto da escala
    cbar.set_label('Probabilidade de Transição', labelpad=15)
    
    # Adicionando títulos e rótulos aos eixos
    plt.title("Matriz de Transição de Estados (Cadeia de Markov)", pad=15)
    plt.xlabel("Próximo Estado $(t)$",labelpad=10)
    plt.ylabel("Estado Atual $(t-1)$",labelpad=10)
    
    # Rotacionando os labels dos eixos
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Ajusta o layout para evitar cortes
    plt.tight_layout()
    
    # Salva e exibe a figura
    # (Certifique-se de que a variável 'pasta_saida' já está definida no seu código)
    plt.savefig(f'{pasta_saida}/Matriz_de_Transição.png', format='png', dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
matriz_P

In [ ]:
mask_NaNs = st['G'].isna()
blocos = (mask_NaNs != mask_NaNs.shift()).cumsum()
st_gaps = st[mask_NaNs].groupby(blocos).apply(
    lambda x: pd.Series({
        'Inicio': x.index.min(),
        'Fim': x.index.max(),
        'Duracao_min': len(x) }) ).reset_index(drop=True)
st_gaps = st_gaps.sort_values(by='Duracao_min', ascending=False)

st_gaps.head(2)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

config_visual = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 17,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'mathtext.fontset': 'stix'}
        
with plt.rc_context(config_visual):
    if not st_gaps.empty:
        maior_gap = st_gaps.iloc[1]
        inicio_gap = maior_gap['Inicio']
        fim_gap = maior_gap['Fim']
        duracao = maior_gap['Duracao_min']
        
        margem = pd.Timedelta(hours=2)
        janela_inicio = inicio_gap - margem
        janela_fim = fim_gap + margem
        
        st_zoom = st.loc[janela_inicio:janela_fim]
        plt.figure(figsize=(12, 6))
        
        # 1. Curva de Céu Claro (Referência)
        plt.plot(st_zoom.index, st_zoom['ClearSky_Tracker'], 
                 label='Céu Claro (Tracker)', color='orange', linestyle='--', alpha=0.8)
        
        # 2. Curva Original (Vermelha) - Onde o NaN virou 0, a linha despenca e sobe
        plt.plot(st_zoom.index, st_zoom['G_NaNs__Trocado_Por_Zeros'], 
                 label='Medição Original (Falhas = 0)', color='red', linewidth=1.5, alpha=0.6)
    
        # 3. A LINHA PRETA ADICIONADA: Traçada cirurgicamente no nível zero, apenas durante a falha
        plt.hlines(y=0, xmin=inicio_gap, xmax=fim_gap, color='black', 
                   linewidth=2.5, zorder=5, label='Falha do Sensor (Sinal = 0)')

        # 4. A CURVA DE MARKOV: Plotada apenas no trecho exato da falha para não cobrir o vermelho externo
        trecho_imputado = st_zoom.loc[inicio_gap:fim_gap, 'G_Imputado']
        plt.plot(trecho_imputado.index, trecho_imputado, 
                 label='Imputação Markov', color='blue', linewidth=2, alpha=0.9, zorder=6)
    
        plt.title(f'Preenchimento de Falhas do Sinal de Irradiação Por Cadeias De Markov \n' 
                  f'$gap$ de {duracao} minutos ({(duracao/60).round(2)} Horas) em {inicio_gap.strftime("%d/%m/%Y")}')
        
        plt.ylabel('Irradiância Global no Plano do Tracker (W/m²)')
        plt.xlabel('Tempo (Local)')
        
        plt.legend()
        plt.tight_layout()

        pasta_saida = './GRAFICOS'
        if not os.path.exists(pasta_saida):
            os.makedirs(pasta_saida)

        plt.savefig(f'{pasta_saida}/Preenchimento_de_Falhas_de_Irradiacao_Por_Cadeias_De_Markov.png', format='png', dpi=600, bbox_inches='tight')
        plt.show()
    else:
        print("Nenhuma falha diurna encontrada na base de dados para plotar.")
    plt.close()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

config_visual = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 17,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'mathtext.fontset': 'stix'
}
        
with plt.rc_context(config_visual):
    if not st_gaps.empty:
        # Puxando o 10º maior gap da sua lista (índice 9)
        maior_gap = st_gaps.iloc[9]
        inicio_gap = maior_gap['Inicio']
        fim_gap = maior_gap['Fim']
        duracao = maior_gap['Duracao_min']
        
        margem = pd.Timedelta(hours=2)
        janela_inicio = inicio_gap - margem
        janela_fim = fim_gap + margem
        
        st_zoom = st.loc[janela_inicio:janela_fim]
        plt.figure(figsize=(12, 6))
        
        # 1. Curva de Céu Claro (Referência)
        plt.plot(st_zoom.index, st_zoom['ClearSky_Tracker'], 
                 label='Céu Claro (Curva ideal de Irradiãncia)', color='orange', linestyle='--', alpha=0.8)
        
        # 2. Curva Original (Onde o NaN virou 0, a linha despenca e sobe em vermelho)
        plt.plot(st_zoom.index, st_zoom['G_NaNs__Trocado_Por_Zeros'], 
                 label='Medição Real dio Sensor', color='red', linewidth=1.5, alpha=0.6)

        # 3. A LINHA PRETA: Traçada cirurgicamente no nível zero, apenas durante a falha
        plt.hlines(y=0, xmin=inicio_gap, xmax=fim_gap, color='black', 
                   linewidth=2.5, zorder=5, label='Falha do Sensor (Sinal = 0)')
    
        # 4. A CURVA DE MARKOV: Plotada apenas no trecho exato da falha (usando .loc)
        trecho_imputado = st_zoom.loc[inicio_gap:fim_gap, 'G_Imputado']
        plt.plot(trecho_imputado.index, trecho_imputado, 
                 label='Imputação Estocástica Por Cadeias de Markov', color='blue', linewidth=2, alpha=0.9, zorder=6)
    
        plt.title(f'Imputação Estocástica via Cadeias de Markov e Simulção de Monte Carlo no Sinal de Irradiância\n' 
                  f'$gap$ de {duracao} minutos ({(duracao/60).round(2)} Horas) em {inicio_gap.strftime("%d/%m/%Y")}')
        
        plt.ylabel('Irradiância Global no Plano do Tracker (W/m²)')
        plt.xlabel('Mês-Dia Hora')
        
        plt.legend()
        plt.tight_layout()

        pasta_saida = './GRAFICOS'
        if not os.path.exists(pasta_saida):
            os.makedirs(pasta_saida)

        plt.savefig(f'{pasta_saida}/Preenchimento de Falhas de Irradiacao Por Cadeias De Markov.png', 
                    format='png', dpi=600, bbox_inches='tight')
        plt.show()
    else:
        print("Nenhuma falha diurna encontrada na base de dados para plotar.")
    plt.close()

In [ ]:
Welch=MetodoDeWelch()
Welch.AplicaMetodoDeWelch(df=st, coluna_potencia='G_NaNs__Trocado_Por_Zeros', coluna_iradiancia='G_Imputado', 
                          win_size=7*1440, NOME_COLUNA_DE_POTENCIA_PARA_PLOT='G_NaNs__Trocado_Por_Zeros',
                          NOME_COLUNA_DE_IRRADIANCIA_PARA_PLOT='G_Cadeia_de_Markov', minuto_correção=0)

In [ ]:


# 2. Gerando o novo gráfico de verificação
dia_teste = '2019-11-01' 
#'2019-01-03' 
# '2019-02-10' 
# '2019-02-21'
# '2019-02-20'

# Filtrando os dados apenas para este dia
st_dia = st.loc[dia_teste]

plt.figure(figsize=(12, 6))
plt.plot(st_dia.index, st_dia['G'], label='Medição Real (G)', color='blue')

# A MUDANÇA ESTÁ AQUI: Plotando a curva horizontal (ClearSky) em vez da POA
plt.plot(st_dia.index, st_dia['ClearSky_Tracker'], label='Céu Claro Teórico (Horizontal)', color='orange', linestyle='--')

plt.title(f'Comparação GHI Medido vs Céu Claro Horizontal - {dia_teste}')
plt.ylabel('Irradiância (W/m²)')
plt.legend()
plt.grid(True)
plt.show()